# DdT_1_vol_11 — general MEI checks and corpus publication review

This notebook calls `scripts/validate_corpus.py`: installed CAMAT runs its general checks, then this repository applies `validation/profile.json` and `validation/rules.py` to the explicit work list. It does not install/update CAMAT or rewrite MEI.

Read three separate results: **MEI conformance**, **consumer diagnostics**, and **DdT publication acceptance**. A general pass is useful evidence, but source comparison and publication sign-off still belong to editors. The initial manifest contains only Op. I/1; other files in the directory are shown as unlisted and are not silently certified.

Prepare the environment once in a terminal: `python scripts/update_camat.py`. Add `--pull` only when you want to fast-forward the clean local CAMAT checkout on its current branch. To install CAMAT from Git instead, use `--git-ref v0.2.7`, the tested release. Also install libxml2's `xmllint`. Use the same Python environment as this kernel, and restart the kernel after updating CAMAT.

The tracked CSVs beside the scores are shared editorial material. Refresh all 17 with `python scripts/refresh_editorial_reports.py`; this records portable provenance in `validation/editorial-reports.json` and isolates native consumers per score.


In [ ]:
from pathlib import Path
import importlib.util
import sys
import pandas as pd
from IPython.display import display, FileLink
import camat
from camat import camat_provenance
from camat.mei_consistency_workflow import load_report, report_summary, source_snippet

ROOT = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
             if (p / "validation/works.json").is_file()
             and (p / "scripts/validate_corpus.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Start the notebook inside DdT_1_vol_11.")
spec = importlib.util.spec_from_file_location("ddt_corpus_validation", ROOT / "scripts/validate_corpus.py")
corpus = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = corpus
spec.loader.exec_module(corpus)
identity = camat_provenance()
print("Python:", sys.executable)
print("CAMAT module/distribution:", identity["module_version"], "/", identity.get("distribution_version"))
print("CAMAT source:", identity["module_path"])
print("CAMAT revision:", identity.get("git_branch"), identity.get("git_commit"))


## 1. Select the work list and options

`WORK_KEYS=None` selects the entire explicit manifest, currently only the pilot. Use a list such as `["op1_01_sonata_f_major"]` for a subset. Add other works to the manifest with their own sources, catalogue and staff roles before expanding publication checks; the pilot's five staves are not a corpus-wide default.

**Review** accepts working filenames and reports outstanding publication work. **Release** requires final identity, review, hashes and metadata, and forces full Verovio rendering plus CAMAT parsing. No reviewer sign-off or warning waiver is fabricated. In particular, the official external-IIIF `surface/@corresp` warnings remain pending review.

Keep `RUN_CHECKS=False` to inspect the plan without running checks. Consumer options can be enabled in review. Image networking is an independent HTTP accessibility check; unavailable scans do not establish invalid MEI.

For another assembled score, choose `validation/editorial-works.json` and its single work key. This list is review-only. For all shared reports, use the batch helper. The current CAMAT parser crashes on Appendix I; the helper records that failure without terminating the whole refresh.


In [ ]:
MODE = "review"                    # "review" or "release"
MANIFEST = ROOT / "validation/works.json"  # or validation/editorial-works.json for other assembled works
PROFILE = ROOT / "validation/profile.json"
WORK_KEYS = ["op1_01_sonata_f_major"]    # None selects the entire explicit manifest
REPORT_DIR = ROOT / "validation/runs/notebook"

CHECK_VEROVIO = True                # full load/render; forced in release
CHECK_CAMAT_PARSE = True            # event-table probe; forced in release
PARSING_BACKEND = "verovio"
CHECK_NETWORK = True               # optional HTTP image targets only
CHECK_SCHEMATRON = True             # disabling leaves conformance incomplete
CHECK_PPQ = True                    # compare timing; never strip attributes
CHECK_EDITORIAL_DIAGNOSTICS = True  # extra duration/layer/term checks
CHECK_GROUP_DIAGNOSTICS = True      # staff/term comparisons in the selected score(s)
RUN_CHECKS = False


In [ ]:
plan = corpus.plan_corpus_validation(
    root=ROOT, manifest=MANIFEST, profile=PROFILE, work_keys=WORK_KEYS,
    mode=MODE, output_dir=REPORT_DIR,
)
print("Mode:", MODE)
print("Scope:", plan["manifest"].get("scope"))
print("DdT profile:", plan["profile"]["id"], plan["profile"]["profile_version"])
print("Schema:", plan["schema"])
print("Reference mode: standalone")
print("Required general passes: XML/references, RELAX NG, Schematron, resources, input hashes")
print("Consumers:", "Verovio", CHECK_VEROVIO or MODE == "release", "; CAMAT", CHECK_CAMAT_PARSE or MODE == "release")
print("HTTP image connectivity:", CHECK_NETWORK)
display(pd.DataFrame([{"work": w["key"], "file": w["file"],
                       "exists": (ROOT / w["file"]).is_file(),
                       "editorial_status": w.get("editorial_status")} for w in plan["works"]]))
print("Unlisted work-directory MEI (not checked):", len(plan["unlisted_files"]))
for path in plan["unlisted_files"]:
    print(" ", path.relative_to(ROOT) if path.is_relative_to(ROOT) else path)
print("Reports:", REPORT_DIR)
print("Run enabled:", RUN_CHECKS)


## 2. Run and read the results

**XML/references** checks readable XML, unique IDs and supported local pointers. **RELAX NG** checks allowed MEI structure/values with `xmllint`. **Schematron** executes relationships embedded in that same official CMN 5.1 schema using SchXslt/SaxonC: a structurally valid `harm` without a starting anchor can fail here. An `xml-model` declaration alone executes no validator. CMN means Common Music Notation; DdT policy is an additional layer.

**Resources** checks supplied local images and declared geometry. **Input hashes** verify the complete run leaves selected MEI unchanged. **Musical diagnostics** provide meter, staff/layer, duration, pitch and optional PPQ review aids. **Verovio/CAMAT** report consumer behavior separately; parsing probes nonempty event tables rather than proving every musical event is correct. **Image network** can be `not-applicable` with an explanation when no HTTP images are linked.

**DdT rules** check this edition's schema declarations, scoped digital license, work/file/source identity, responsibility links, addressable event IDs, per-work staff roles, primary facsimile/page/measure relationships and annotations. Release adds final metadata, recorded input hashes and human review. They cannot verify the musical truth of an editorial argument.

The execution table counts checks; its `findings` column counts emitted rows. A dash means no count was available. The finding summary groups rows by severity/category/check, so group counts differ from check counts and CSV row counts. `passed` may still have advisory findings; `skipped` means disabled; `execution-error` means a pass could not finish. Missing required validators cannot pass. A review exit code of zero with `needs-review` is not publication approval.


In [ ]:
validation = None
report_df = None
if not RUN_CHECKS:
    print("Skipped. Review the plan, then set RUN_CHECKS = True.")
else:
    validation = corpus.run_corpus_validation(
        root=ROOT, manifest=MANIFEST, profile=PROFILE, work_keys=WORK_KEYS,
        mode=MODE, output_dir=REPORT_DIR,
        check_verovio=CHECK_VEROVIO, check_camat_parse=CHECK_CAMAT_PARSE,
        parsing_backend=PARSING_BACKEND, check_network=CHECK_NETWORK,
        check_schematron=CHECK_SCHEMATRON, check_ppq=CHECK_PPQ,
        check_editorial_diagnostics=CHECK_EDITORIAL_DIAGNOSTICS,
        check_group_diagnostics=CHECK_GROUP_DIAGNOSTICS,
    )
    corpus.print_validation_summary(validation, REPORT_DIR)
    print("CLI-equivalent exit code:", corpus.corpus_exit_code(validation))
    execution_rows = [{"check": name, "requested": entry["requested"],
                       "required": entry["required"], "status": entry["status"],
                       "findings": entry.get("findings_count", "—")}
                      for name, entry in validation.record["executions"].items()]
    print("General checks: required = needed by this mode's gate")
    display(pd.DataFrame(execution_rows))
    print("DdT policy execution:", validation.record["ddt"]["policy_execution_status"])
    print("DdT publication findings:", validation.record["ddt"]["publication_findings_count"])
    report_df = load_report(REPORT_DIR / "findings.csv")
    print(f"Finding summary: {len(report_df)} individual rows, grouped below")
    display(report_summary(report_df))
    print("Selected MEI unchanged:", all(item["unchanged"] for item in validation.record["inputs"]))


## 3. Open findings and inspect context

The top-level **Findings CSV/JSON** contain the same combined rows: CAMAT findings plus DdT findings, distinguished by `origin` and `check`. **Run JSON** contains execution/provenance, schema/profile/work-list/source hashes and both repository identities.

The files under `general/` retain the CAMAT-only findings and progress record; their CSV intentionally has fewer rows than the combined CSV when DdT adds findings. They are not duplicate score-named exports. `general/schematron/` retains extracted rules, compiled validator and per-input SVRL.

This notebook replaces `REPORT_DIR`'s current reports on each run. Keep the complete folder or choose a fresh directory to preserve evidence. For a stalled native pass, inspect incomplete progress in `general/run.json`; automatic hard timeouts are not yet implemented.


In [ ]:
if validation is not None:
    for label, path in [
        ("Run JSON: execution, versions and hashes", REPORT_DIR / "run.json"),
        ("Findings CSV: CAMAT + DdT", REPORT_DIR / "findings.csv"),
        ("Findings JSON: same combined findings", REPORT_DIR / "findings.json"),
        ("General-only CSV: CAMAT findings", REPORT_DIR / "general/findings.csv"),
        ("General progress/run JSON", REPORT_DIR / "general/run.json"),
    ]:
        print(label)
        display(FileLink(str(path.relative_to(ROOT) if path.is_relative_to(ROOT) else path)))
    display(report_df.head(100))


In [ ]:
ROW_INDEX = None
if validation is not None and ROW_INDEX is not None:
    print(source_snippet(report_df.loc[ROW_INDEX], root=ROOT, active_files=plan["files"], radius=4))
else:
    print("Set ROW_INDEX to a findings row to inspect its source context.")


In [ ]:
# Refresh the tracked editorial CSV for a single selected work.
if validation is None:
    print("No report to copy. Enable RUN_CHECKS and run validation first.")
elif len(plan["works"]) != 1:
    print("Use python scripts/refresh_editorial_reports.py to refresh all shared per-work reports.")
else:
    report_spec = importlib.util.spec_from_file_location("ddt_editorial_reports", ROOT / "scripts/refresh_editorial_reports.py")
    reports = importlib.util.module_from_spec(report_spec)
    report_spec.loader.exec_module(reports)
    work = plan["works"][0]
    dest = reports.report_path(ROOT, work)
    reports.write_csv(dest, reports.editorial_rows(validation, work, ROOT))
    print("Wrote", dest.relative_to(ROOT))
    print("Before committing reports, run python scripts/refresh_editorial_reports.py to refresh the complete provenance.")


## 4. Compare the score with its source

Refresh the shared viewer from the local CAMAT checkout with `python scripts/sync_camat_viewer.py`, then open `mei_facsimile_viewer.ipynb`. It uses the installed CAMAT library and a corpus-relative pilot path; syncing the notebook does not update the package.

Edit in mei-friend, inspect in the viewer, and rerun these checks. Conversion/cleanup remains explicit: `camat.convert_harm_startid_to_tstamp`, `camat.link_pb_to_surface` and `camat.mei_consistency_workflow` provide the old reusable operations, which validation never invokes.

See `validation/README.md` for commands and review/release semantics. The retired local tools are preserved under `docs/archive/local-tools/artifacts/` with their original bytes, outside the active workflow.
